# CDD Notebook 12 — Stage 3b: Probe-Circuit Alignment (CPA)

This notebook trains a logistic probe on each component and computes CPA between the probe direction and the top causal direction.

Expected headline result: mean CPA = 0.0666 with probe accuracy = 0.9145.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np
from tqdm.auto import tqdm

CPA_results = {}

for comp, acts in tqdm(component_activations.items(), desc='CPA calculation'):
    T = np.array(acts['truthful'])
    H = np.array(acts['hallucinated'])
    X = np.vstack([T, H])
    y = np.array([1]*len(T) + [0]*len(H))
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    clf = LogisticRegression(C=100, max_iter=1000, solver='lbfgs')
    clf.fit(X_scaled, y)
    probe_dir  = clf.coef_[0] / (np.linalg.norm(clf.coef_[0]) + 1e-8)
    causal_dir = causal_directions[comp][0]
    causal_dir = causal_dir / (np.linalg.norm(causal_dir) + 1e-8)
    cpa = float(np.abs(np.dot(causal_dir, probe_dir)))
    CPA_results[comp] = {'CPA': cpa, 'probe_acc': float(clf.score(X_scaled, y))}

mean_CPA       = np.mean([v['CPA'] for v in CPA_results.values()])
mean_probe_acc = np.mean([v['probe_acc'] for v in CPA_results.values()])
print(f'Mean CPA (causal-probe alignment) : {mean_CPA:.4f}')
print(f'Mean probe accuracy               : {mean_probe_acc:.4f}')